In [23]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Download essential natural language processing resources from NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

print("All dependencies imported and NLTK resources downloaded successfully.")

All dependencies imported and NLTK resources downloaded successfully.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rosha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rosha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rosha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [24]:
# Read the CSV file into a pandas DataFrame
df = pd.read_csv(r"C:\Users\rosha\Downloads\new_combined_dirty_production_dataset.csv")

# Display shape properties and category distribution to verify data balance
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("--- Class Distribution ---")
print(df['category'].value_counts())

# Print a preview of the raw input data rows
print("\n--- First 3 Rows Preview ---")
print(df.head(3))


Dataset Shape: 420 rows, 2 columns

--- Class Distribution ---
category
TECHNICAL    105
GENERAL      105
BILLING      105
HR           105
Name: count, dtype: int64

--- First 3 Rows Preview ---
                                                text   category
0  We are experiencing production database timeou...  TECHNICAL
1  Where can I read your official updated privacy...    GENERAL
2  Hello, I am reaching out hoping you can help m...  TECHNICAL


In [25]:
# Initialize NLTK utilities
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Create an empty list to collect the fully preprocessed text samples
cleaned_texts_list = []

print("Starting explicit text preprocessing pipeline across all rows...")

# Iterate sequentially through every raw string in the dataframe
for raw_text in df['text']:
    # 1. Lowercase configuration
    text_lower = raw_text.lower()
    
    # 2. Noise Removal (Strip out everything except standard English alphabet characters)
    text_alpha_only = re.sub(r'[^a-zA-Z\s]', '', text_lower)
    
    # 3. Explicit Tokenisation (Break the raw string into single word tokens)
    tokens = word_tokenize(text_alpha_only)
    
    # 4 & 5. Combined Stopword Filtering and Lemmatisation
    processed_tokens = []
    for token in tokens:
        if token not in stop_words:
            # Map word down to its primary root dictionary structure
            root_lemma = lemmatizer.lemmatize(token)
            processed_tokens.append(root_lemma)
            
    # Reconstruct the tokens back into a single unified sentence string
    clean_sentence = " ".join(processed_tokens)
    cleaned_texts_list.append(clean_sentence)

# Assign the processed lists back into a distinct column in the dataframe
df['cleaned_text'] = cleaned_texts_list

print("Preprocessing complete! New column 'cleaned_text' populated.")
print("\nTransformation Sample View:")
print(f"Original Text:  {df['text'].iloc[0]}")
print(f"Cleaned Text:   {df['cleaned_text'].iloc[0]}")


Starting explicit text preprocessing pipeline across all rows...
Preprocessing complete! New column 'cleaned_text' populated.

Transformation Sample View:
Original Text:  We are experiencing production database timeouts during peak operational hours. (ID Token: #5045)
Cleaned Text:   experiencing production database timeouts peak operational hour id token


In [26]:
df

,text,category,cleaned_text
0,We are experiencing production database timeou...,TECHNICAL,experiencing production database timeouts peak...
1,Where can I read your official updated privacy...,GENERAL,read official updated privacy policy user data...
2,"Hello, I am reaching out hoping you can help m...",TECHNICAL,hello reaching hoping help clarify issue exper...
3,Can you provide a simple overview of what serv...,GENERAL,provide simple overview service enterprise pla...
4,Where is your corporate headquarters physical ...,GENERAL,corporate headquarters physical street address...
...,...,...,...
415,Can u plzz process a refund for invoice number...,BILLING,u plzz process refund invoice number id token ...
416,The text font rendering on the settings portal...,TECHNICAL,text font rendering setting portal look comple...
417,When will the updated payroll schedules and ta...,HR,updated payroll schedule tax form released yea...
418,"Hi support team, hope ur having an amazing day...",GENERAL,hi support team hope ur amazing day current ce...


In [27]:
# Define features (X) and target label classes (y) using the preprocessed text column
X = df['cleaned_text']
y = df['category']

# Execute an 80/20 train-test split, using stratify to maintain equal category distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training subset sample size: {X_train.shape[0]} rows")
print(f"Testing validation sample size: {X_test.shape[0]} rows")


Training subset sample size: 336 rows
Testing validation sample size: 84 rows


In [28]:
# Initialize the Vectorizer, setting it to evaluate individual words and word pairs (unigrams/bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1, 1))

# Learn the vocabulary dictionary and transform the training features into mathematical matrices
X_train_vectors = vectorizer.fit_transform(X_train)

# Transform the testing features against the learned training matrix structures
X_test_vectors = vectorizer.transform(X_test)

print(f"Text Vectorisation Completed.")
print(f"Training Feature Matrix Dimensions: {X_train_vectors.shape} (Rows, Extracted Unique Features)")


Text Vectorisation Completed.
Training Feature Matrix Dimensions: (336, 342) (Rows, Extracted Unique Features)


In [36]:
# Instantiate a Logistic Regression model optimized for clean, multi-class classification
classifier_model = LogisticRegression(C=0.01,max_iter=1000, random_state=42)

print("Training the Logistic Regression classifier model...")
classifier_model.fit(X_train_vectors, y_train)
print("Model training execution complete.")


Training the Logistic Regression classifier model...
Model training execution complete.


In [37]:
# Execute predictions against the unseen test vectors
y_predictions = classifier_model.predict(X_test_vectors)

# Generate and print the explicit machine learning classification matrix report
print("=================== PERFORMANCE METRICS REPORT ===================")
print(classification_report(y_test, y_predictions))

print("=================== CONFUSION MATRIX ===================")
labels_order = classifier_model.classes_
matrix_output = confusion_matrix(y_test, y_predictions, labels=labels_order)

# Display a clean formatted matrix view
matrix_df = pd.DataFrame(matrix_output, index=labels_order, columns=labels_order)
print("Rows = True Classes | Columns = Predicted Classes")
print(matrix_df)


=================== PERFORMANCE METRICS REPORT ===================
              precision    recall  f1-score   support

     BILLING       1.00      1.00      1.00        21
     GENERAL       0.95      1.00      0.98        21
          HR       1.00      0.95      0.98        21
   TECHNICAL       1.00      1.00      1.00        21

    accuracy                           0.99        84
   macro avg       0.99      0.99      0.99        84
weighted avg       0.99      0.99      0.99        84

=================== CONFUSION MATRIX ===================
Rows = True Classes | Columns = Predicted Classes
           BILLING  GENERAL  HR  TECHNICAL
BILLING         21        0   0          0
GENERAL          0       21   0          0
HR               0        1  20          0
TECHNICAL        0        0   0         21


In [38]:
# --- STEP A: Ingest Live Sample Raw Input ---
new_raw_ticket = "URGENT request: My team is getting a critical 500 error on the platform UI dashboard and it is completely down! Please fix this ASAP."

print(f"Incoming Ticket Content:\n\"{new_raw_ticket}\"\n")

# --- STEP B: Hardcoded Urgency Keyword Check ---
urgency_keywords = ["urgent", "asap", "down", "broken", "emergency", "crashing", "fail"]
ticket_lower = new_raw_ticket.lower()
is_urgent_priority = any(keyword in ticket_lower for keyword in urgency_keywords)
priority_tag = "HIGH" if is_urgent_priority else "NORMAL"

# --- STEP C: Live Preprocessing Sequence ---
live_alpha = re.sub(r'[^a-zA-Z\s]', '', ticket_lower)
live_tokens = word_tokenize(live_alpha)
live_cleaned_tokens = [lemmatizer.lemmatize(t) for t in live_tokens if t not in stop_words]
live_processed_text = " ".join(live_cleaned_tokens)

# --- STEP D: Edge Case Safety Handling (Empty Strings check) ---
if not live_processed_text.strip():
    predicted_category = "UNKNOWN"
    confidence_percentage = "0.00%"
    final_routing = "HUMAN_REVIEW_QUEUE"
    requires_manual_triage = True
    fallback_note = "Edge case met: The message contains zero meaningful words after stopword cleaning."
else:
    # --- STEP E: Transform and Compute ML Probabilities ---
    live_vector = vectorizer.transform([live_processed_text])
    probabilities = classifier_model.predict_proba(live_vector)
    available_classes = classifier_model.classes_
    
    # Extract the highest scoring output
    highest_prob_index = probabilities.argmax()
    predicted_category = available_classes[highest_prob_index]
    confidence_value = float(probabilities[0][highest_prob_index])
    confidence_percentage = f"{confidence_value * 100:.2f}%"
    
    # --- STEP F: Fallback Queue Checking (60% Confidence Threshold) ---
    confidence_threshold = 0.60
    if confidence_value < confidence_threshold:
        final_routing = "HUMAN_REVIEW_QUEUE"
        requires_manual_triage = True
        fallback_note = "Low confidence fallback metric triggered."
    else:
        final_routing = predicted_category
        requires_manual_triage = False
        fallback_note = "Automated routing conditions met successfully."

# --- STEP G: Print Production Metadata Payload ---
print("=================== LIVE TRIAGE ENGINE OUTPUT ===================")
print(f"1. Priority Severity Tag:   {priority_tag}")
print(f"2. Raw Predicted Label:     {predicted_category}")
print(f"3. Algorithmic Confidence:   {confidence_percentage}")
print(f"4. Ultimate Assigned Route:  {final_routing}")
print(f"5. Requires Manual Triage:  {requires_manual_triage}")
print(f"6. Routing Status Notes:     {fallback_note}")


Incoming Ticket Content:
"URGENT request: My team is getting a critical 500 error on the platform UI dashboard and it is completely down! Please fix this ASAP."

=================== LIVE TRIAGE ENGINE OUTPUT ===================
1. Priority Severity Tag:   HIGH
2. Raw Predicted Label:     TECHNICAL
3. Algorithmic Confidence:   25.51%
4. Ultimate Assigned Route:  HUMAN_REVIEW_QUEUE
5. Requires Manual Triage:  True
6. Routing Status Notes:     Low confidence fallback metric triggered.


In [41]:
!pip install joblib

import joblib

In [42]:

# FINAL STEP: PIPELINE EXPORT AND MODEL SERIALIZATION

# Bundle the vectorizer weights and model states into a dictionary object
production_pipeline_payload = {
    'vectorizer_vocabulary': vectorizer,
    'trained_classifier_weights': classifier_model,
    'lemmatizer_utility': lemmatizer,
    'stop_words_utility': stop_words
}

# Export the bundle binary file locally to your machine
joblib.dump(production_pipeline_payload, "optimized_triage_pipeline_model.joblib")

print("=================== PIPELINE ARTIFACT EXPORT COMPLETE ===================")
print("Saved file: 'optimized_triage_pipeline_model.joblib'")
print("This unified binary can now be loaded instantly into any live web API framework!")


=================== PIPELINE ARTIFACT EXPORT COMPLETE ===================
Saved file: 'optimized_triage_pipeline_model.joblib'
This unified binary can now be loaded instantly into any live web API framework!


# With more data, my next step would be moving from raw text keywords to a transformer model like BERT. This would let the system understand context much better, which is crucial for customers who write in with multiple problems at once. I'd also reconfigure it to support multi-label routing so tickets don't get trapped in just one department. Finally, using a wider pool of real-world messy emails would let us optimize the 60% confidence threshold to prevent manual triage bottlenecks."